In [1]:
from datasets import load_dataset

filenames = ['meta_Electronics', 'meta_Cell_Phones_and_Accessories']

full_ds = load_dataset("milistu/AMAZON-Products-2023-Arabic", split="train")  # one full download, not streamed
filtered = full_ds.filter(
    lambda batch: [f in filenames for f in batch["filename"]],
    batched=True,
)

/home/ahmed/projects/Arabic-shopping-assistant/Shopping-Assistant/src/shopping-assistant/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Filter: 100%|██████████| 117243/117243 [00:03<00:00, 38545.17 examples/s]


In [2]:
filtered.save_to_disk("./assets/electronics_cellphones")

Saving the dataset (1/1 shards): 100%|██████████| 12743/12743 [00:15<00:00, 819.38 examples/s]


In [1]:
from datasets import load_from_disk

dataset = load_from_disk("assets/electronics_cellphones")

/home/ahmed/projects/Arabic-shopping-assistant/Shopping-Assistant/src/shopping-assistant/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset

Dataset({
    features: ['parent_asin', 'date_first_available', 'title', 'title_arb', 'description', 'description_arb', 'filename', 'main_category', 'categories', 'store', 'average_rating', 'rating_number', 'price', 'features', 'details', 'embeddings', 'image'],
    num_rows: 12743
})

In [3]:
columns_to_keep = ['parent_asin', 'title', 'description', 'filename', 'store', 'average_rating', 'rating_number', 'price', 'image']
filtered = dataset.select_columns(columns_to_keep)
filtered

Dataset({
    features: ['parent_asin', 'title', 'description', 'filename', 'store', 'average_rating', 'rating_number', 'price', 'image'],
    num_rows: 12743
})

In [4]:
def build_embedding_text(batch):
    return {
        "embedding_text": [
            f"title:{title}. \ndescription: {description or ''}".strip()
            for title, description in zip(batch["title"], batch["description"])
        ]
    }
filtered = filtered.map(build_embedding_text, batched=True)

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-albert-small-v2")



Loading weights: 100%|██████████| 25/25 [00:00<00:00, 502.94it/s]


In [9]:
to_embed = filtered["embedding_text"]  # plain list of strings, no .numpy needed

embeddings = model.encode(to_embed, batch_size=32, show_progress_bar=True)
print(embeddings.shape)

Batches: 100%|██████████| 399/399 [4:47:35<00:00, 43.25s/it]      


(12743, 768)


In [17]:
filtered = filtered.add_column("title_description_embeddings", embeddings.tolist())

In [19]:
filtered.save_to_disk("./assets/products_with_embeddings")

Saving the dataset (1/1 shards): 100%|██████████| 12743/12743 [00:02<00:00, 5339.58 examples/s]
